## Introduction and Definitions

This will be an exploration data analysis of the different physical characteristics of stars in the near Milky Way galaxy.

We will make use of the Hipparcos catalog as our main dataset. Hipparcos is a european astronomy satellite that was operational between from 1989 to 1993 and recorded observations of ~118k stellar objects. The dataset is publicly available from multiple sources, including the Stasbourg Astronomical Data Center (https://cds.unistra.fr/).

Unfortunately there's a catch. Space is huge, and it's hard to see faraway objects in the dark. As a result the Hipparcos catalog is heavily biased towards more luminous stars, since they're easier to see. So we're not going to get a very good representative sample of lower-luminosity objects unless we bring in another dataset.

Luckily we have very descriptively named "Woolley Catalog of Stars within 25 Parsecs" to supplement. This won't feature many of the bright stars found in hipparcos, but it will have a much higer proportion of dim stars and should give us enough of a sample size to work with. 

We will quantify the approximate ranges of several characteristics across different stellar classifications:

**Direct Measurements**
- Color (B-V)
- Apparent Magnitude

**Calculated Measures**
- Temperature (Derived from Color)
- Luminosity or Absolute Magnitude (Derived from Apparent Magnitude and Distance)
- Size (in solar radii)
- Mass (in solar masses)

Analysis will be done across two categories of classification, **Spectral Class** and **Luminosity Class**

Spectral Class is denoted by a letter that represents the approximate temperature of the star, from hottest to coolest they are (O, B, A, F, G, K, and M). The wavelength of thermal radiation (light) is inversly proportional to the temperature emmiting it, so hot stars will appear more blue and cooler stars more red. These categories are then subdivided 0-9 for the more specific temperature classification (0 is hottest, 9 is coolest) 

Luminosity Class is determined by it's luminosity (quantity of emitted radiation) as well as it's temperature. Note that temperature is relevant to both classifications - this will be important later. The classes are:
- Ia:   Hypergiants
- Ib:   Supergiants
- II:   Bright Giants
- III:  Giants
- IV:   Subgiants
- V:    Main Sequence / Red Dwarfs
- VI:   Subdwarfs
- VII:  White Dwarfs

The Sun, for example, is a relatively hot main sequence star and is classified as a type **G2V**. Polaris (the North Star) is actually a trinary system composed of the F7Ib Supergiant Polaris A and it's two smaller companions (F3V and F6V)

Ok enough trivia. Now that you understand the basics lets get down to business...

In [10]:
import urllib.request
import csv
import sqlite3
import io


def download_and_load_data(url, db, target_table):
    print("Downloading...")
    with urllib.request.urlopen(url) as resp:
        raw = resp.read().decode("utf-8")

    
    lines = [l for l in raw.splitlines() if not l.startswith("#")]
    reader = csv.DictReader(io.StringIO("\n".join(lines)))

    cols = reader.fieldnames
    expected = len(cols)
    print(f"Columns ({expected}): {cols}")

    rows = []
    for i, row in enumerate(reader):
        # DictReader puts overflow columns under None key — drop them
        row.pop(None, None)
        if len(row) != expected:
            print(f"Skipping row {i}: got {len(row)} cols")
            continue
        rows.append([row[c] for c in cols])

    print(f"Parsed {len(rows)} valid rows")

    col_defs = ", ".join(f'"{c.strip().replace('-', '_').lower()}" TEXT' for c in cols)
    placeholders = ", ".join("?" * expected)

    con = sqlite3.connect(db)

    # Try except so if an error occurs we don't have to restart kernal because of a connection left open
    try:
        cur = con.cursor()
        cur.execute(f'DROP TABLE IF EXISTS "{target_table}"')
        cur.execute(f'CREATE TABLE "{target_table}" ({col_defs})')
        cur.executemany(f'INSERT INTO "{target_table}" VALUES ({placeholders})', rows)
        con.commit()
        con.close()
        print(f"Loaded {len(rows)} rows into {db} table '{target_table}'")
    except Exception as e:
        print(e)
        con.close()


In [11]:
URL_HIPPARCOS = (
    "https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync"
    "?REQUEST=doQuery&LANG=ADQL&FORMAT=csv"
    '&QUERY=SELECT+*+FROM+"I/239/hip_main"'
)

DB= 'stellar_analysis.db'
TARGET_TABLE_HIP = 'hipparcos_main'

download_and_load_data(URL_HIPPARCOS, DB, TARGET_TABLE_HIP)

Downloading...
Columns (79): ['recno', 'HIP', 'Proxy', 'RAhms', 'DEdms', 'Vmag', 'VarFlag', 'r_Vmag', 'RAICRS', 'DEICRS', 'AstroRef', 'Plx', 'pmRA', 'pmDE', 'e_RAICRS', 'e_DEICRS', 'e_Plx', 'e_pmRA', 'e_pmDE', 'DE:RA', 'Plx:RA', 'Plx:DE', 'pmRA:RA', 'pmRA:DE', 'pmRA:Plx', 'pmDE:RA', 'pmDE:DE', 'pmDE:Plx', 'pmDE:pmRA', 'F1', 'F2', 'BTmag', 'e_BTmag', 'VTmag', 'e_VTmag', 'm_BTmag', 'B-V', 'e_B-V', 'r_B-V', 'V-I', 'e_V-I', 'r_V-I', 'CombMag', 'Hpmag', 'e_Hpmag', 'Hpscat', 'o_Hpmag', 'm_Hpmag', 'Hpmax', 'HPmin', 'Period', 'HvarType', 'moreVar', 'morePhoto', 'CCDM', 'n_CCDM', 'Nsys', 'Ncomp', 'MultFlag', 'Source', 'Qual', 'm_HIP', 'theta', 'rho', 'e_rho', 'dHp', 'e_dHp', 'Survey', 'Chart', 'Notes', 'HD', 'BD', 'CoD', 'CPD', '(V-I)red', 'SpType', 'r_SpType', '_RA_icrs', '_DE_icrs']
Parsed 118218 valid rows
Loaded 118218 rows into stellar_analysis.db table 'hipparcos_main'


In [14]:
import pandas as pd

# Quick sanity check to make sure everything worked correctly
con = sqlite3.connect(DB)
df = pd.read_sql('SELECT * FROM hipparcos_main', con)
con.close()

# Lots of different stellar designations to parse out
print(df['sptype'].unique().tolist()[:30])

['F5', 'K3V', 'B9', 'F0V', 'G8III', 'M0V:', 'G0', 'M6e-M8.5e Tc', 'G5', 'F6V', 'A2', 'K4III', 'K0III', 'K0', 'K2', 'F3V', ' ', 'K5', 'G8/K0III/IV', 'F2V', 'G0V', 'G3IV', 'F7V', 'G5V', 'F3/F5V', 'A0', 'B8', 'F2', 'F7.5IV-V', 'G6V']


In [ ]:
from stellar_parser import parse_stellar_column

# TY Claude I owe you my life
def build_silver_table(db_path, source_table_name, table_name, source_cols):
    con = sqlite3.connect(db_path)

    # Load only the columns we need from source
    src_cols_sql = ", ".join(f'"{c}"' for c in source_cols)
    df = pd.read_sql(f"SELECT {src_cols_sql} FROM {source_table_name}", con)

    # Convert column names to lowercase and SQL-safe
    df.columns = [c.lower().replace("-", "_") for c in df.columns]
    col_types = {k.lower().replace("-", "_"): v for k, v in source_cols.items()}

    # Cast columns to their specified types
    for col, dtype in col_types.items():
        if dtype == "REAL":
            # Strip whitespace and non-numeric characters (preserve decimal points)
            df[col] = df[col].astype(str).str.strip().str.replace(r"[^\d.]", "", regex=True)
            df[col] = pd.to_numeric(df[col], errors="coerce")

        elif dtype == "INTEGER":
            df[col] = df[col].astype(str).str.strip().str.replace(r"[^\d]", "", regex=True)
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        # TEXT columns need no conversion

    # Parse Sp -> sp_class, lum_class
    parsed = parse_stellar_column(df["sptype"])
    df["sp_class"] = parsed["spectral_type"]
    df["lum_class"] = parsed["lum_class"]

    # Sort by plx descending (closest stars first); nulls last
    df.sort_values("plx", ascending=False, na_position="last", inplace=True)

    # Write silver table
    df.to_sql(table_name, con, if_exists="replace", index=False)

    cur = con.cursor()
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_{table_name}_sp_class ON {table_name}(sp_class)')
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_{table_name}_sp_plx ON {table_name}(sp_class, plx DESC)')
    cur.execute(f"ALTER TABLE {table_name} ADD COLUMN src TEXT DEFAULT '{source_table_name.split('_')[0]}'")
    con.commit()

    row_count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table_name}", con).iloc[0]["n"]
    print(f"Created '{table_name}' with {row_count:,} rows.")
    con.close()
